# Student Dropout Prediction: Multi-Model Comparison

This notebook implements and compares five different machine learning classifiers to predict student dropout using stratified 10-fold cross-validation. The approach focuses on experimentation and baseline model performance comparison without hyperparameter tuning.

## Models Compared
- **Logistic Regression** - Linear baseline with feature scaling
- **Decision Tree** - Interpretable non-linear model
- **Random Forest** - Ensemble method for robust predictions
- **XGBoost** - Gradient boosting ensemble method
- **Neural Network (MLP)** - Non-linear deep learning approach

## Table of Contents

1. **[Import Libraries & Load Data](#1-import-libraries--load-data)** - Load necessary Python packages and dataset
2. **[Data Preparation](#2-data-preparation)** - Split data and prepare features and target variables for cross-validation
3. **[Model Configuration](#3-model-configuration)** - Configure all four models with balanced class weights
4. **[Cross-Validation Evaluation](#4-cross-validation-evaluation)** - Train and evaluate all models using 10-fold CV
5. **[Model Comparison Dashboard](#5-model-comparison-dashboard)** - Compare performance metrics across models
6. **[Unified ROC Analysis](#6-unified-roc-analysis)** - Visualize all models' ROC curves together
7. **[Feature Importance Comparison](#7-feature-importance-comparison)** - Compare feature importance across models
8. **[Best Model Selection & Comprehensive Analysis](#8-best-model-selection--comprehensive-analysis)** - Select best model, analyze performance, and provide recommendations
9. **[Summary](#summary)** - Key findings and next steps

## Key Benefits of This Comparison
- **Cross-Validation Focus**: All models evaluated with rigorous 10-fold cross-validation methodology
- **Direct Comparison**: Side-by-side performance metrics and visualizations
- **Model Selection**: Data-driven approach to choosing the best model based on CV performance
- **Feature Insights**: Understanding which features are important across different algorithms
- **Unified ROC Curves**: All four models displayed on a single ROC plot for easy comparison

## 1. Import Libraries & Load Data

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')
import joblib

# Scikit-learn imports
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    roc_auc_score, roc_curve, make_scorer
)
from sklearn.utils.class_weight import compute_class_weight

# XGBoost import
import xgboost as xgb
from xgboost import XGBClassifier

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Load the dataset
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nDataset info:")
print(df.info())

## 2. Data Preparation

**Purpose**: Split data into train/validation sets and prepare features and target variables for cross-validation.

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()
test_df = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

In [ ]:
# Define target and feature columns
exclude_cols = ['set', target_col]
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Target column: {target_col}")
print(f"Number of features: {len(feature_cols)}")
print(f"Excluded columns: {exclude_cols}")

# Prepare feature matrices and target vectors
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

# Combine train and validation for cross-validation
X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

print(f"\nCombined train+val shape for cross-validation: {X_train_val.shape}")

print("\nClass distribution in combined train+val set:")
print(y_train_val.value_counts(normalize=True))

In [ ]:
# Verify data quality (there should be no missing values)
missing_counts = X_train_val.isnull().sum().sum()
print(f"Missing values in features: {missing_counts}")
if missing_counts > 0:
    print("⚠️ Warning: Missing values found! Please perform imputation.")
    missing_cols = X_train_val.isnull().sum()[X_train_val.isnull().sum() > 0]
    print("Columns with missing values:")
    print(missing_cols)
else:
    print("✅ Data quality verified: No missing values found.")

## 3. Model Configuration

**Purpose**: Configure all five models with appropriate parameters and balanced class weights.

In [ ]:
# Calculate baseline accuracy
baseline_accuracy = y_train_val.value_counts(normalize=True).max()
print(f"Baseline accuracy (majority class): {baseline_accuracy:.3f}")

In [ ]:
# Calculate scale_pos_weight for class imbalance (XGBoost alternative to class_weight='balanced')
negative_samples = (y_train_val == 0).sum()
positive_samples = (y_train_val == 1).sum()
scale_pos_weight = negative_samples / positive_samples

# Calculate neural network class weights (for class imbalance)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_val), y=y_train_val)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

# Configure all models with balanced class weights
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        class_weight='balanced',
        max_iter=1000
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced',
        max_depth=12,               # Moderate depth limit
        min_samples_split=30,       # Moderate split threshold
        min_samples_leaf=15,        # Moderate leaf threshold
        max_features='sqrt',        # Standard feature limitation
        min_impurity_decrease=0.005 # Small improvement requirement
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
        max_depth=12,           # Moderate depth limit
        min_samples_split=30,   # Moderate split threshold
        min_samples_leaf=15,    # Moderate leaf threshold
        max_features='sqrt',    # Default feature limit
        max_samples=0.9         # Use 90% of samples
    ),
    'XGBoost': XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        verbosity=0,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        max_depth=5,                        # Moderate depth limit
        learning_rate=0.08,                 # Slightly slower learning
        n_estimators=80,                    # Moderate number of trees
        reg_alpha=0.3,                      # Moderate L1 regularization
        reg_lambda=1.5,                     # Moderate L2 regularization
        subsample=0.85,                     # Use 85% of samples
        colsample_bytree=0.85,              # Use 85% of features
        min_child_weight=3                  # Moderate leaf requirement
        ),
    'Neural Network': MLPClassifier(
    random_state=42,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1
    )
}

print("Model configurations:")
for name, model in models.items():    
    print(f"- {name}: {type(model).__name__}")

## 4. Cross-Validation
**Purpose**: Train and evaluate all models using stratified 10-fold cross-validation.

In [ ]:
# Set up cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring metrics
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision'
}

print(f"Cross-validation strategy: {cv_strategy.n_splits}-fold stratified")
print(f"Scoring metrics: {list(scoring_metrics.keys())}")

In [ ]:
# Initialize results storage
cv_results = {}

print("Starting cross-validation for all models...\n")

for model_name, model in models.items():
    print(f"Evaluating {model_name}...")
    start_time = time.time()
    
    # Extract data
    X_for_cv = X_train_val.values
    
    # Perform cross-validation
    cv_scores = cross_validate(
        model, X_for_cv, y_train_val,
        cv=cv_strategy,
        scoring=scoring_metrics,
        n_jobs=-1,
        return_train_score=True
    )
    
    # Store results
    cv_results[model_name] = cv_scores
    
    end_time = time.time()

print("Cross-validation completed for all models!")

## 5. Model Comparison Dashboard

**Purpose**: Create comprehensive visualizations and tables comparing all models' cross-validation performance, including overfitting analysis and stability assessment.

In [ ]:
# Create performance comparison table
comparison_data = []

for model_name in models.keys():
    scores = cv_results[model_name]
    
    comparison_data.append({
        'Model': model_name,
        'Recall': f"{scores['test_recall'].mean():.3f}",
        'F1 Score': f"{scores['test_f1'].mean():.3f}",
        'ROC AUC': f"{scores['test_roc_auc'].mean():.3f}",
        'Precision': f"{scores['test_precision'].mean():.3f}",
        'Accuracy': f"{scores['test_accuracy'].mean():.3f}",
        'Accuracy vs Baseline': f"{(scores['test_accuracy'].mean() - baseline_accuracy):.3f}",
    })

comparison_df = pd.DataFrame(comparison_data)
print("Model Performance Comparison:")
print("=" * 120)
print(comparison_df.to_string(index=False))
print("=" * 120)

In [ ]:
# Create comprehensive cross-validation performance visualization
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle('Cross-Validation Performance: All Models Comparison', fontsize=16, fontweight='bold')

model_names_list = list(models.keys())

# Extract cross-validation metrics
cv_accuracies = [cv_results[name]['test_accuracy'].mean() for name in model_names_list]
cv_roc_aucs = [cv_results[name]['test_roc_auc'].mean() for name in model_names_list]
cv_recalls = [cv_results[name]['test_recall'].mean() for name in model_names_list]
cv_f1s = [cv_results[name]['test_f1'].mean() for name in model_names_list]
cv_precisions = [cv_results[name]['test_precision'].mean() for name in model_names_list]

cv_accuracy_stds = [cv_results[name]['test_accuracy'].std() for name in model_names_list]
cv_roc_auc_stds = [cv_results[name]['test_roc_auc'].std() for name in model_names_list]
cv_recall_stds = [cv_results[name]['test_recall'].std() for name in model_names_list]
cv_f1_stds = [cv_results[name]['test_f1'].std() for name in model_names_list]
cv_precision_stds = [cv_results[name]['test_precision'].std() for name in model_names_list]

# Plot 1: Recall (Row 1, Col 1)
bars1 = axes[0, 0].bar(model_names_list, cv_recalls, yerr=cv_recall_stds, capsize=5, alpha=0.8, color='lightgreen')
axes[0, 0].set_title('Recall', fontweight='bold')
axes[0, 0].set_ylabel('Recall')
axes[0, 0].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars1, cv_recalls, cv_recall_stds):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: F1 Score (Row 1, Col 2)
bars2 = axes[0, 1].bar(model_names_list, cv_f1s, yerr=cv_f1_stds, capsize=5, alpha=0.8, color='plum')
axes[0, 1].set_title('F1 Score', fontweight='bold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars2, cv_f1s, cv_f1_stds):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 3: Precision (Row 2, Col 1)
bars3 = axes[1, 0].bar(model_names_list, cv_precisions, yerr=cv_precision_stds, capsize=5, alpha=0.8, color='salmon')
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars3, cv_precisions, cv_precision_stds):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 4: Accuracy (Row 2, Col 2)
bars4 = axes[1, 1].bar(model_names_list, cv_accuracies, yerr=cv_accuracy_stds, capsize=5, alpha=0.8, color='skyblue')
axes[1, 1].axhline(y=baseline_accuracy, color='red', linestyle='--', alpha=0.7, label=f'Baseline: {baseline_accuracy:.3f}')
axes[1, 1].set_title('Accuracy', fontweight='bold')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars4, cv_accuracies, cv_accuracy_stds):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 5: ROC AUC (Row 3, Col 1)
bars5 = axes[2, 0].bar(model_names_list, cv_roc_aucs, yerr=cv_roc_auc_stds, capsize=5, alpha=0.8, color='lightcoral')
axes[2, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Random: 0.500')
axes[2, 0].set_title('ROC AUC', fontweight='bold')
axes[2, 0].set_ylabel('ROC AUC')
axes[2, 0].legend()
axes[2, 0].tick_params(axis='x', rotation=45)
for bar, val, std in zip(bars5, cv_roc_aucs, cv_roc_auc_stds):
    axes[2, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 6: Train vs Validation Accuracy (Row 3, Col 2)
cv_train_accuracies = [cv_results[name]['train_accuracy'].mean() for name in model_names_list]
x_pos = np.arange(len(model_names_list))
width = 0.35
bars6_train = axes[2, 1].bar(x_pos - width/2, cv_train_accuracies, width, label='Training', alpha=0.8, color='steelblue')
bars6_val = axes[2, 1].bar(x_pos + width/2, cv_accuracies, width, label='Validation', alpha=0.8, color='darkorange')
axes[2, 1].set_title('Training vs Validation Accuracy (Overfitting Check)', fontweight='bold')
axes[2, 1].set_ylabel('Accuracy')
axes[2, 1].set_xticks(x_pos)
axes[2, 1].set_xticklabels(model_names_list, rotation=45)
axes[2, 1].legend()
for i, (train_acc, val_acc) in enumerate(zip(cv_train_accuracies, cv_accuracies)):
    gap = train_acc - val_acc
    axes[2, 1].text(i, max(train_acc, val_acc) + 0.02, f'Gap: {gap:.3f}', ha='center', va='bottom', fontsize=9, color='red' if gap > 0.05 else 'green')

plt.tight_layout()
plt.show()

## Try out models on test data

In [ ]:
# TEST SET EVALUATION - FINAL MODEL PERFORMANCE ASSESSMENT
from sklearn.metrics import recall_score, precision_score, f1_score

# Prepare test data
X_test = test_df[feature_cols]
y_test = test_df[target_col]

# Train final models on full train+val set for test evaluation
test_final_models = {}

for model_name, model in models.items():
    # Extract final training data
    X_final = X_train_val.values
    
    # Train the model
    trained_model = model.fit(X_final, y_train_val)
    test_final_models[model_name] = trained_model

# Dictionary to store test results
test_results = {}

# Evaluate each model on test set
for model_name, model in test_final_models.items():
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Calculate ROC AUC if possible
    try:
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_proba = model.decision_function(X_test)
        roc_auc = roc_auc_score(y_test, y_proba)
    except:
        roc_auc = None
    
    # Store results
    test_results[model_name] = {
        'Accuracy': accuracy,
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'ROC_AUC': roc_auc
    }

# Create test results summary table
print(f"\n" + "=" * 80)
print("TEST SET PERFORMANCE SUMMARY")
print("=" * 80)

test_df_results = pd.DataFrame(test_results).T
test_df_results = test_df_results.round(4)

print(test_df_results.to_string())

# Compare with cross-validation results
print(f"\n" + "=" * 80)
print("TEST vs CROSS-VALIDATION PERFORMANCE COMPARISON")
print("=" * 80)

comparison_results = []
for model_name in test_results.keys():
    if model_name in cv_results:
        cv_accuracy = cv_results[model_name]['test_accuracy'].mean()
        cv_recall = cv_results[model_name]['test_recall'].mean()
        
        test_accuracy = test_results[model_name]['Accuracy']
        test_recall = test_results[model_name]['Recall']
        
        accuracy_diff = test_accuracy - cv_accuracy
        recall_diff = test_recall - cv_recall
        
        comparison_results.append({
            'Model': model_name,
            'CV_Accuracy': cv_accuracy,
            'Test_Accuracy': test_accuracy,
            'Accuracy_Diff': accuracy_diff,
            'CV_Recall': cv_recall,
            'Test_Recall': test_recall,
            'Recall_Diff': recall_diff
        })

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.round(4)

print("CV vs Test Performance (Diff = Test - CV):")
print(comparison_df.to_string(index=False))

# Performance analysis
print(f"\n" + "=" * 60)
print("TEST SET ANALYSIS & INSIGHTS")
print("=" * 60)

# Find best performing models on test set
best_accuracy_model = test_df_results['Accuracy'].idxmax()
best_recall_model = test_df_results['Recall'].idxmax()
best_f1_model = test_df_results['F1'].idxmax()

print(f"\nBest Performance on Test Set:")
print(f"  🎯 Best Accuracy: {best_accuracy_model} ({test_df_results.loc[best_accuracy_model, 'Accuracy']:.4f})")
print(f"  🎯 Best Recall: {best_recall_model} ({test_df_results.loc[best_recall_model, 'Recall']:.4f})")
print(f"  🎯 Best F1 Score: {best_f1_model} ({test_df_results.loc[best_f1_model, 'F1']:.4f})")

# Generalization assessment
print(f"\nGeneralization Assessment:")
for _, row in comparison_df.iterrows():
    model_name = row['Model']
    acc_diff = row['Accuracy_Diff']
    recall_diff = row['Recall_Diff']
    
    if abs(acc_diff) < 0.02:
        generalization = "✅ Excellent"
    elif abs(acc_diff) < 0.05:
        generalization = "🟡 Good"
    else:
        generalization = "⚠️  Concerning"
    
    print(f"  • {model_name}: {generalization} (Accuracy: {acc_diff:+.3f}, Recall: {recall_diff:+.3f})")

## 6. Unified ROC Analysis

**Purpose**: Visualize all models' ROC curves on a single plot for direct comparison.

In [ ]:
# Generate ROC curves on TEST DATA using already trained models
plt.figure(figsize=(12, 10))

# Define colors for each model
colors = ['blue', 'green', 'red', 'orange', 'purple']

# Use the already trained models from the test evaluation section
final_models = test_final_models

# Prepare test data for ROC evaluation
X_test_roc = test_df[feature_cols].values
y_test_roc = test_df[target_col]

print("Generating ROC curves on test data using already trained models...")
print("-" * 70)

for i, (model_name, model) in enumerate(final_models.items()):
    print(f"Generating test ROC curve for {model_name}...")
    
    # Get prediction probabilities on TEST data
    if hasattr(model, 'predict_proba'):
        y_proba_test = model.predict_proba(X_test_roc)[:, 1]
    else:
        y_proba_test = model.decision_function(X_test_roc)
    
    # Calculate ROC curve on TEST data
    fpr, tpr, _ = roc_curve(y_test_roc, y_proba_test)
    roc_auc_test = roc_auc_score(y_test_roc, y_proba_test)
    
    # Plot ROC curve
    plt.plot(fpr, tpr, color=colors[i], lw=3, 
             label=f'{model_name} (Test AUC = {roc_auc_test:.3f})')

# Add diagonal line for random classifier
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', alpha=0.8, 
         label='Random Classifier (AUC = 0.500)')

# Customize plot
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves on Test Data: All Models Comparison', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTest Data ROC Analysis Summary:")
print("-" * 40)
for i, (model_name, model) in enumerate(final_models.items()):
    # Calculate test AUC for summary
    if hasattr(model, 'predict_proba'):
        y_proba_test = model.predict_proba(X_test_roc)[:, 1]
    else:
        y_proba_test = model.decision_function(X_test_roc)
    test_auc = roc_auc_score(y_test_roc, y_proba_test)
    
    if test_auc >= 0.8:
        performance = "🟢 Excellent"
    elif test_auc >= 0.7:
        performance = "🟡 Good"
    elif test_auc >= 0.6:
        performance = "🟠 Fair"
    else:
        performance = "🔴 Poor"
    
    print(f"{model_name}: Test AUC = {test_auc:.3f} {performance}")

## 7. Feature Importance Comparison

**Purpose**: Compare feature importance/coefficients across all models to understand which features are consistently important.

In [ ]:
# Extract feature importance/coefficients from each model
feature_importance = pd.DataFrame({
    'Feature': feature_cols
})

for model_name, model in final_models.items():
    if model_name == 'Logistic Regression':
        # Get coefficients (absolute values for importance)
        importance = np.abs(model.coef_[0])
        feature_importance[f'{model_name}_Importance'] = importance
        
    elif model_name in ['Decision Tree', 'Random Forest', 'XGBoost']:
        # Get feature importance
        importance = model.feature_importances_
        feature_importance[f'{model_name}_Importance'] = importance
        
    elif model_name == 'Neural Network':
        # For neural networks, we'll use a simple approximation
        # This is less interpretable but gives some indication
        if hasattr(model, 'coefs_'):
            # Sum of absolute weights from input to first hidden layer
            importance = np.abs(model.coefs_[0]).sum(axis=1)
            feature_importance[f'{model_name}_Importance'] = importance

# Normalize importance scores to 0-1 scale for comparison
importance_cols = [col for col in feature_importance.columns if 'Importance' in col]
for col in importance_cols:
    if col in feature_importance.columns:
        max_val = feature_importance[col].max()
        if max_val > 0:
            feature_importance[col] = feature_importance[col] / max_val

# Calculate average importance across models
feature_importance['Average_Importance'] = feature_importance[importance_cols].mean(axis=1)

# Sort by average importance
feature_importance = feature_importance.sort_values('Average_Importance', ascending=False)

In [ ]:
# Visualize feature importance comparison
fig, axes = plt.subplots(2, 3, figsize=(24, 16))
fig.suptitle('Feature Importance Comparison Across Models', fontsize=16, fontweight='bold')

# Get top 10 features for visualization
top_10_features = feature_importance.head(10)

# Plot each model's feature importance
model_cols = [col for col in feature_importance.columns if 'Importance' in col and 'Average' not in col]

for i, col in enumerate(model_cols[:5]):  # Limit to 5 models
    row = i // 3
    col_idx = i % 3
    
    # Create horizontal bar plot
    y_pos = np.arange(len(top_10_features))
    axes[row, col_idx].barh(y_pos, top_10_features[col], alpha=0.7)
    axes[row, col_idx].set_yticks(y_pos)
    axes[row, col_idx].set_yticklabels([name[:30] for name in top_10_features['Feature']], fontsize=10)
    axes[row, col_idx].set_xlabel('Normalized Importance', fontsize=11)
    axes[row, col_idx].set_title(col.replace('_Importance', ''), fontsize=12, fontweight='bold')
    axes[row, col_idx].grid(axis='x', alpha=0.3)
    
    # Invert y-axis to show most important at top
    axes[row, col_idx].invert_yaxis()

# Hide the empty subplot (position [1,2] for 6th subplot)
if len(model_cols) == 5:
    axes[1, 2].set_visible(False)

plt.tight_layout()
plt.show()

## 8. Best Model Selection & Comprehensive Analysis

**Purpose**: Select the best performing model based on:
- Recall
- F1
- ROC-AUC

In [ ]:
# COMPREHENSIVE MODEL COMPARISON AND BEST MODEL SELECTION
print("COMPREHENSIVE MODEL COMPARISON ANALYSIS")
print("=" * 80)
print(f"Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Dataset: {df.shape[0]:,} students, {len(feature_cols)} features")
print(f"Evaluation Method: Stratified 10-fold cross-validation")

# 1. MODEL RANKINGS AND SELECTION
print("\n1. MODEL RANKINGS BASED ON CROSS-VALIDATION:")
print("-" * 50)

model_rankings_cv = []
for model_name in models.keys():
    scores = cv_results[model_name]
    
    model_rankings_cv.append({
        'Model': model_name,
        'CV_ROC_AUC_Mean': scores['test_roc_auc'].mean(),
        'CV_ROC_AUC_Std': scores['test_roc_auc'].std(),
        'CV_Accuracy_Mean': scores['test_accuracy'].mean(),
        'CV_Accuracy_Std': scores['test_accuracy'].std(),
        'CV_Recall_Mean': scores['test_recall'].mean(),
        'CV_Recall_Std': scores['test_recall'].std(),
        'CV_F1_Mean': scores['test_f1'].mean(),
        'CV_F1_Std': scores['test_f1'].std(),
        'Stability_Score': 1 / (scores['test_roc_auc'].std() + 0.001),
        'Improvement_Over_Baseline': scores['test_accuracy'].mean() - baseline_accuracy,
        # Combined score: average of recall, f1, and roc_auc
        'Combined_Score': (scores['test_recall'].mean() + scores['test_f1'].mean() + scores['test_roc_auc'].mean()) / 3
    })

cv_ranking_df = pd.DataFrame(model_rankings_cv)
# Rank by combined score
cv_ranking_df = cv_ranking_df.sort_values(['Combined_Score'], ascending=[False])

print("Rankings based on Combined Score (Average of Accuracy and ROC AUC):")

# Display rankings with performance and stability
for i, row in cv_ranking_df.iterrows():
    rank = list(cv_ranking_df.index).index(i) + 1
    model = row['Model']
    auc = row['CV_ROC_AUC_Mean']
    auc_std = row['CV_ROC_AUC_Std']
    acc = row['CV_Accuracy_Mean']
    acc_std = row['CV_Accuracy_Std']
    recall = row['CV_Recall_Mean']
    recall_std = row['CV_Recall_Std']
    f1 = row['CV_F1_Mean']
    f1_std = row['CV_F1_Std']
    combined_score = row['Combined_Score']
    improvement = row['Improvement_Over_Baseline']
    
    # Stability assessment
    if auc_std < 0.02 and acc_std < 0.02:
        stability = "Very Stable"
        stability_emoji = "✅"
    elif auc_std < 0.04 and acc_std < 0.04:
        stability = "Moderately Stable"
        stability_emoji = "🟡"
    else:
        stability = "Less Stable"
        stability_emoji = "⚠️"
    
    emoji = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "📊"
    print(f"{emoji} {rank}. {model}:")
    print(f"     Combined Score: {combined_score:.3f}")
    print(f"     ROC AUC: {auc:.3f} (±{auc_std:.3f}) | Accuracy: {acc:.3f} (±{acc_std:.3f})")
    print(f"     Recall: {recall:.3f} (±{recall_std:.3f}) | F1 Score: {f1:.3f} (±{f1_std:.3f})")
    print(f"     Improvement over baseline: +{improvement:.3f} | Stability: {stability_emoji} {stability}")

# Select best model based on combined score
best_model_name = cv_ranking_df.iloc[0]['Model']

print(f"\n🏆 BEST MODEL SELECTED: {best_model_name}")
print(f"   • Combined Score: {cv_ranking_df.iloc[0]['Combined_Score']:.3f}")
print(f"   • CV ROC AUC: {cv_ranking_df.iloc[0]['CV_ROC_AUC_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_ROC_AUC_Std']:.3f})")
print(f"   • CV Accuracy: {cv_ranking_df.iloc[0]['CV_Accuracy_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_Accuracy_Std']:.3f})")
print(f"   • CV Recall: {cv_ranking_df.iloc[0]['CV_Recall_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_Recall_Std']:.3f})")
print(f"   • CV F1 Score: {cv_ranking_df.iloc[0]['CV_F1_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_F1_Std']:.3f})")
print(f"   • Improvement over baseline: +{cv_ranking_df.iloc[0]['Improvement_Over_Baseline']:.3f}")

# Store the ranking dataframe for later use
ranking_df = cv_ranking_df.copy()
ranking_df.columns = ['Model', 'ROC_AUC_Mean', 'ROC_AUC_Std', 'Accuracy_Mean', 'Accuracy_Std', 'Recall_Mean', 'Recall_Std', 'F1_Mean', 'F1_Std', 'Stability_Score', 'Improvement_Over_Baseline', 'Combined_Score']

In [ ]:
# 2. DETAILED ANALYSIS OF BEST MODEL
print(f"\n2. DETAILED ANALYSIS OF BEST MODEL: {best_model_name}")
print("-" * 60)

# Get cross-validation performance for comprehensive analysis
best_cv_scores = cv_results[best_model_name]

print("\nPerformance Metrics:")
print(f"  • Validation Accuracy: {best_cv_scores['test_accuracy'].mean():.4f} (±{best_cv_scores['test_accuracy'].std():.4f})")
print(f"  • Training Accuracy: {best_cv_scores['train_accuracy'].mean():.4f} (±{best_cv_scores['train_accuracy'].std():.4f})")
print(f"  • Validation ROC AUC: {best_cv_scores['test_roc_auc'].mean():.4f} (±{best_cv_scores['test_roc_auc'].std():.4f})")
print(f"  • Validation Recall: {best_cv_scores['test_recall'].mean():.4f} (±{best_cv_scores['test_recall'].std():.4f})")
print(f"  • Validation F1 Score: {best_cv_scores['test_f1'].mean():.4f} (±{best_cv_scores['test_f1'].std():.4f})")

# Overfitting and stability analysis
train_accuracy = best_cv_scores['train_accuracy'].mean()
cv_accuracy = best_cv_scores['test_accuracy'].mean()
cv_roc_auc = best_cv_scores['test_roc_auc'].mean()
cv_recall = best_cv_scores['test_recall'].mean()
cv_f1 = best_cv_scores['test_f1'].mean()
accuracy_gap = train_accuracy - cv_accuracy
acc_std = best_cv_scores['test_accuracy'].std()
auc_std = best_cv_scores['test_roc_auc'].std()

print(f"\nModel Quality Assessment:")
print(f"  • Training-Validation gap: {accuracy_gap:+.4f}")

# Assess overfitting
if accuracy_gap < 0.02:
    overfitting = "✅ No overfitting detected"
elif accuracy_gap < 0.05:
    overfitting = "🟡 Slight overfitting"
else:
    overfitting = "⚠️  Potential overfitting"

# Assess stability
if acc_std < 0.02 and auc_std < 0.02:
    stability = "✅ Very stable"
elif acc_std < 0.04 and auc_std < 0.04:
    stability = "🟡 Moderately stable"
else:
    stability = "⚠️  Less stable"

# Performance rating
if cv_roc_auc >= 0.8:
    auc_rating = "Excellent"
elif cv_roc_auc >= 0.7:
    auc_rating = "Good"
elif cv_roc_auc >= 0.6:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

# Recall assessment for class imbalance
if cv_recall >= 0.8:
    recall_rating = "Excellent"
elif cv_recall >= 0.7:
    recall_rating = "Good"
elif cv_recall >= 0.6:
    recall_rating = "Fair"
else:
    recall_rating = "Poor"

print(f"  • Overfitting: {overfitting}")
print(f"  • Stability: {stability}")
print(f"  • Performance Rating: {auc_rating} (ROC AUC: {cv_roc_auc:.3f})")
print(f"  • Recall Rating: {recall_rating} (Recall: {cv_recall:.3f}) - Important for identifying at-risk students")
print(f"  • Improvement over baseline: {(cv_accuracy - baseline_accuracy)*100:.1f} percentage points")

# Model configuration details
print(f"\nModel Configuration:")
if best_model_name in final_models:
    best_model = final_models[best_model_name]
    print(f"  • Type: {type(best_model).__name__}")
    key_params = ['random_state', 'class_weight', 'n_estimators', 'max_depth', 'max_iter', 'early_stopping']
    params = best_model.get_params()
    relevant_params = {k: v for k, v in params.items() if k in key_params}
    if relevant_params:
        print(f"  • Key Parameters: {', '.join([f'{k}={v}' for k, v in relevant_params.items()])}")

# Store variables for backward compatibility
best_scores = best_cv_scores

In [ ]:
# 3. KEY INSIGHTS AND PATTERNS
print(f"\n3. KEY INSIGHTS AND PATTERNS:")
print("-" * 40)

# Insight 1: Best performing model type
if best_model_name == 'Random Forest':
    insight1 = "Ensemble methods (Random Forest) perform best, indicating complex feature interactions."
elif best_model_name == 'Neural Network':
    insight1 = "Neural networks capture non-linear patterns effectively for this problem."
elif best_model_name == 'Logistic Regression':
    insight1 = "Linear relationships are sufficient; simpler models work well."
else:
    insight1 = "Tree-based models provide good interpretability with solid performance."

print(f"• Model Type: {insight1}")

# Insight 2: Performance level and readiness
if cv_roc_auc >= 0.8:
    performance_level = "excellent"
    actionable = "Model is ready for further validation and potential deployment."
elif cv_roc_auc >= 0.7:
    performance_level = "good"
    actionable = "Model shows promise but could benefit from feature engineering."
else:
    performance_level = "moderate"
    actionable = "Consider collecting additional features or trying different approaches."

print(f"• Performance Level: Model achieves {performance_level} cross-validation performance.")
print(f"• Readiness: {actionable}")

# Insight 3: Feature importance consistency
if len(importance_cols) > 1:
    avg_importance_corr = feature_importance[importance_cols].corr().values[np.triu_indices_from(
        feature_importance[importance_cols].corr().values, k=1)].mean()
    
    if avg_importance_corr > 0.7:
        consistency_insight = "High agreement between models on feature importance suggests robust feature selection."
    elif avg_importance_corr > 0.4:
        consistency_insight = "Moderate agreement between models on feature importance."
    else:
        consistency_insight = "Low agreement suggests different models focus on different aspects of the data."
    
    print(f"• Feature Consistency: {consistency_insight}")

print(f"\n4. TOP PREDICTIVE FEATURES:")
print("-" * 40)
top_20_features = feature_importance.head(20)
for i, (_, row) in enumerate(top_20_features.iterrows(), 1):
    feature_name = row['Feature'][:50]  # Truncate long names
    avg_importance = row['Average_Importance']
    print(f"{i}. {feature_name}: {avg_importance:.3f}")

print(f"\n5. RECOMMENDATIONS AND NEXT STEPS:")
print("-" * 40)

print(f"📋 Immediate Actions:")
if cv_roc_auc >= 0.75:
    print(f"   ✅ Proceed with hyperparameter tuning for {best_model_name}")
    print(f"   ✅ Consider final validation on hold-out test set")
    print(f"   ✅ Prepare for potential deployment pipeline")
else:
    print(f"   ⚠️  Focus on feature engineering before further optimization")
    print(f"   ⚠️  Collect additional features or domain expertise")
    print(f"   ⚠️  Consider alternative modeling approaches")

print(f"\n📊 Model Development Priorities:")
print(f"   • Hyperparameter tuning focus: {best_model_name}")
print(f"   • Use cross-validation for parameter optimization")
print(f"   • Consider ensemble methods if multiple models perform similarly")

print(f"\n🔬 Long-term Improvements:")
print(f"   • Feature engineering based on top predictive features")
print(f"   • Advanced hyperparameter optimization (Bayesian, Grid Search)")
print(f"   • Ensemble of top-performing models")
print(f"   • Additional data collection for underperforming classes")

## Save models and cv results

In [ ]:
# Define data to be saved
analysis_data = {
    'cv_results': cv_results,
    'final_models': final_models,
    'best_model_name': best_model_name,
    'feature_importance': feature_importance,
    'ranking_df': ranking_df,
    'baseline_accuracy': baseline_accuracy
}
# Save data
joblib.dump(analysis_data, PROCESSED_DIR / 'model_comparison_analysis.pkl')
print("\nModel comparison analysis data saved successfully!")